In [0]:
import configparser
config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')

In [0]:
print(config['TABLES']['provider_silver'])

In [0]:
%run 
/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/common/common_utils

In [0]:
%run 
./silver_utils


In [0]:
table_mapping=[]

for table in config['TABLES']:
    if table.endswith('_table') and 'processed_file' not in table:
        source=table.replace('_table',"")

        table_mapping.append(
            {
                'source':source,
                'bronze_table':config['TABLES'][f'{source}_table'],
                'silver_table':config['TABLES'][f'{source}_silver']
            }
        )
print(table_mapping)

In [0]:
from pyspark.sql.functions import max
from datetime import datetime,timezone
for mapping in table_mapping:

    start_time=datetime.now(timezone.utc)
    source_name=mapping.get('source')
    bronze_table=mapping.get('bronze_table')
    silver_table=mapping.get('silver_table')

    print(f'processing {source_name}')
    print(f'bronze table {bronze_table}')
    print(f'silver table {silver_table}')

    try:
        last_watermark=get_last_watermark(source_name=source_name)
        df_read=read_incremental_bronze(bronze_table=bronze_table,last_watermark=last_watermark)
        record_loaded_to_silver_table=df_read.count()
        if record_loaded_to_silver_table==0:
            status="NO_NEW_RECORDS"
            
        else:
            write_silver(df=df_read,silver_table=silver_table)
            latest_ts = (
            df_read.agg(max("ingestion_ts").alias("latest_ts"))
              .first()["latest_ts"]
            )

            update_silver_watermark(
            source_name=source_name,
            latest_ts=latest_ts,
            silver_table=silver_table)
            status='SUCCESS'
        
        end_time=datetime.now(timezone.utc)
        duration=(end_time-start_time).total_seconds()
        print(f'processing completed for {source_name} took {duration}')

        log_ingestion(
            source_name=source_name,
            table_name=silver_table,
            file_name="",
            file_path="",
            status=status,
            records_loaded=record_loaded_to_silver_table,
            error_message='NA',
            start_ts=start_time,
            end_ts=end_time,
            duration_seconds=duration

        )
        

    except Exception as e:
        print(f'error while processing {source_name} got error {e}')
        end_time = datetime.now(timezone.utc)

        duration = (
            end_time - start_time
        ).total_seconds()
        log_ingestion(
            source_name=source_name,
            table_name=silver_table,
            file_name="",
            file_path="",
            status='FAILED',
            records_loaded=0,
            error_message=str(e),
            start_ts=start_time,
            end_ts=end_time,
            duration_seconds=duration
        )

# healthcare_catalog.silver.beneficiary